In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv(r"C:\users\andre\netflix_titles.csv")
print(df.head())


In [ ]:
print(list(df.columns))
print()
print(df.isna().sum())
print()
print(df.info())
print()
print(df.dtypes)

In [ ]:
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')
df = df.dropna(subset=['date_added'])
df.head()


In [ ]:
df['country'] = df['country'].str.strip().str.split(', ')
df = df.explode('country')
df.head()


In [ ]:

df['country'] = df['country'].fillna('Unknown')
df

In [ ]:
#some info to know where to start analysis 
print(df['director'].isna().sum())
print()
print(df['release_year'].agg([max, min]))
print()
print(df.groupby('type').agg(count = ('type', 'count')))

Which countries produce the most Netflix content? We filter out titles with unknown country and count by country. The USA dominates with over 3600 titles — more than three times second-place India.

In [ ]:
#the top 10 countries based on title count
df_known = df[df['country'] != "Unknown"] #filtered the country column with the values needed
countries = df_known.groupby('country').agg(
    title_count = ('title', 'count')
)
#grouped everything by country and counted all the titles for each


title_count = countries.sort_values(by='title_count', ascending=False).reset_index() #reseted the index for easier analysis in case of further change or if i want to grab something specific
print(title_count.head(10)) 

#as we can see down below, the USA leads with a number of 3642, more than 3X the second place India


Which countries produce the most of each category? We take the already filtered data frame from above and count by title. Again we can see that the USA has the highest movie production besides all almost 3X the second one - India. It stars in TV Shows too, taking up the first place. It is interesting that in South Korea based on the production from above, almost 3/4 out of the production, are TV Shows.

In [ ]:
#the top 10 countries based on each type of streaming
title_max = df_known.groupby(['type', 'country']).agg(
    types = ('title', 'count')
    
)
# used multi index grouping to group types and country and displayed how many titles of each category are there


high_count = title_max.sort_values(by='types', ascending=False).reset_index() #sorted all my values so that i can grab what i need from the resulted data frame

#filtered the 10 highest values based on each category 
top_movies = high_count[high_count['type'] == 'Movie'].head(10)
top_tv_show = high_count[high_count['type'] == 'TV Show'].head(10)
print(top_movies)
print()
print(top_tv_show)


How much did Netflix's volume change over the years? First we extract the year from the whole date and count every title by year. We can see that Netflix wasn't really much untill 2016 when it marked an astonishing 472% increase difference from 2015 and added 5X more content that people could view. 

In [ ]:
#How much did netflix's volume change over the years?
df_yearly = df_known.copy()
df_yearly['year_added'] = pd.to_datetime(df_yearly['date_added']).dt.year #changed the output to only take the years out of the whole data frame
yearly_titles = df_yearly.groupby('year_added').agg(
    added_movies = ('title', 'count')
)
#gathered all of the info i needed to complete my output, counted how many movies was netflix adding each year

yearly_titles['volume_difference'] = yearly_titles['added_movies'].pct_change() * 100 #this opperation is to see the actual increase in procentage 
yearly_titles